# clock/15 — order under *dependencies* (qualitative)

The multiline corpus has mutually **independent** steps, so the prompt's listing is the only thing
that can govern the order — and `16` shows it does. Here the steps depend on each other.

Hardcoded prompts, read by eye. No scores, no plots.

### What a usable dependency needs

If `a -> b` means *a depends on b*, then the producer `b` must have two properties:

1. **hard, but not named** — `a` genuinely cannot proceed without `b`'s result, yet `a`'s
   specification never mentions `b`'s variable. Naming it (`` `c` is divisible by 5 ``) makes the
   model emit degenerate code, so the explicit form is out.
2. **decomposable** — `b`'s implementation splits into meaningful sub-steps, so an intermediate
   line-end is a real move-on pivot.

### Why integers failed, and what replaces them

With `check_conditions(n: int)` the producer is a cheap pure one-liner
(`sum(int(d) for d in str(n))`), so **inlining is easier than reuse** — the model just recomputes and
the implicit edge never forms. The fix is a producer that is a *structure*, not a scalar:

    check_conditions(words: list[str])
    1. the number of times each word appears in words      <- built in a loop; several pivots
    2. whether any word appears more than twice            <- "any word" refers to those counts

Rebuilding a counts dict inside each consumer is visibly absurd, and the one-variable-per-condition
rule pushes against it. The strongest variant (`Q3`) makes the consumer depend on the producer's
**process** rather than its value — *how many times the running maximum increased* cannot be phrased
without the running maximum, so inlining would mean duplicating the whole loop.

### Producer listed FIRST

Deliberate. Putting the consumer first creates a listing/dependency conflict, which is what produced
the degenerate code — and that question is **already answered** by the scripted run at `003ac11`:
with a reversed listing the model emitted 100% valid orders, followed the listing 0% of the time,
and still ran τ = +0.65 on the DAG-free pairs. Dependency wins where it constrains; prompt order
governs where it doesn't. Nothing here needs to re-establish that, and producer-first keeps the code
clean and the prefills natural.

### §3: the pivot must sit at a genuinely INCOMPLETE step

Move-on only means something where the clean model would **stay**. A prefill ending at a finished
statement is not a test — clean and patched both move on and agree. Inside the counts loop there are
two real pivots:

```
    a = {}                        <- pivot 1: dict exists, nothing counted
    for w in words:               <- pivot 2: loop opened, body EMPTY (moving on is a syntax error)
        a[w] = a.get(w, 0) + 1    <- what completes a
```

**`Q0` doubles as the transfer check.** The decision mask was trained on `int`, single-line
predicates; these prompts are `list[str]` and chunky. If move-on does not fire on `Q0` — same pivot,
no dependent step — the vector has not transferred and nothing downstream is interpretable.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, re, json, random, logging
sys.path.append("../..")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2,3,4")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
from rich.console import Console, Group
from rich.theme import Theme
from rich.table import Table
from rich.panel import Panel
from rich.text import Text
from rich.box import SIMPLE_HEAVY
from rich.markup import escape
from src.utils import logging_utils, env_utils

logging_utils.configure(level=logging.WARNING)
logger = logging.getLogger(__name__)
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)
RES = env_utils.DEFAULT_RESULTS_DIR

In [2]:
from src.models import ModelandTokenizer
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(model_key=MODEL_NAME, dtype=torch.bfloat16,
                       device_map="auto", attn_implementation="eager")
print(f"n_layer={mt.n_layer}  n_heads={mt.model.config.num_attention_heads}")

2026-08-12 18:41:09 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

n_layer=64  n_heads=40


In [3]:
# ── §1: the prompts — list[str], producer FIRST, implicit consumers ────────
from src.goals.functional import apply_chat_template

ASSIGN_RULE = (
    "For each item, in the order listed, assign the result to its own local variable, using single "
    "lowercase letters in order — `a` for the first item, `b` for the second, and so on. After all "
    "items have been computed, use a SINGLE print statement that, using those variables, reports "
    "each result."
)
MOD_RULE = ("Compute every value inside `check_conditions` (one local variable per item); do not "
            "define any helper functions.")
STUB = "def check_conditions(words: list[str]):\n"

def build(conds):
    body = "\n".join(f"{i+1}. {c}" for i, c in enumerate(conds))
    instr = ("Write a Python program that implements a function `check_conditions` which, given a "
             "list of strings words, computes the following:\n\n" + body + "\n\n"
             + ASSIGN_RULE + "\n" + MOD_RULE + "\n"
             + "Do not include any comments in your code.")
    return apply_chat_template(mt=mt, user_query=instr, incomplete_code=STUB)

# `dep` = intended (consumer, producer) edges, for reading only. Every edge is IMPLICIT: the
# consumer describes a quantity the producer yields, and never names the producer's variable.
PROMPTS = [
    dict(id="Q0_control", dep=[],
         note="TRANSFER CHECK: same multi-line producer, but nothing depends on it",
         conds=["the number of times each word appears in words",
                "whether every word is entirely lowercase",
                "whether words has at least three entries"]),

    dict(id="Q1_counts", dep=[("b", "a")],
         note="b needs the counts a builds — 'any word' is unphraseable without them",
         conds=["the number of times each word appears in words",
                "whether any word appears more than twice",
                "whether every word is entirely lowercase"]),

    dict(id="Q2_two_consumers", dep=[("b", "a"), ("c", "a")],
         note="two consumers share the counts; rebuilding twice would be absurd",
         conds=["the number of times each word appears in words",
                "whether any word appears more than twice",
                "whether the most frequently occurring word is longer than four characters"]),

    dict(id="Q3_process", dep=[("b", "a")],
         note="STRONGEST: b depends on a's PROCESS, not its value — inlining means duplicating the loop",
         conds=["the running maximum word length after each word in words",
                "how many times that running maximum increased",
                "whether every word is entirely lowercase"]),

    dict(id="Q4_positional", dep=[("b", "a")],
         note="positional consumer — b indexes into the ordering a produces",
         conds=["the words sorted by length, shortest first",
                "whether the third of those is longer than four characters",
                "whether every word is entirely lowercase"]),
]
for p in PROMPTS:
    p["prompt"] = build(p["conds"])
print(f"{len(PROMPTS)} prompts: " + ", ".join(p["id"] for p in PROMPTS))
rich_print(Panel(Text(PROMPTS[1]["prompt"].split("<|im_start|>user")[-1].split("<|im_end|>")[0].strip()),
                 title="[bold]Q1_counts[/bold] · instruction as the model sees it",
                 title_align="left", border_style="grey42", padding=(0, 1)))

5 prompts: Q0_control, Q1_counts, Q2_two_consumers, Q3_process, Q4_positional


╭─ Q1_counts · instruction as the model sees it ──────────────────────────────────────────────────────────────────╮
│ Write a Python program that implements a function `check_conditions` which, given a list of strings words,      │
│ computes the following:                                                                                         │
│                                                                                                                 │
│ 1. the number of times each word appears in words                                                               │
│ 2. whether any word appears more than twice                                                                     │
│ 3. whether every word is entirely lowercase                                                                     │
│                                                                                                                 │
│ For each item, in the order listed, assign the result to its own local variable, using single lowercase letters │
│ in order — `a` for the first item, `b` for the second, and so on. After all items have been computed, use a     │
│ SINGLE print statement that, using those variables, reports each result.                                        │
│ Compute every value inside `check_conditions` (one local variable per item); do not define any helper           │
│ functions.                                                                                                      │
│ Do not include any comments in your code.                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [4]:
# ── §2: generate and read the order + whether the implicit edge was TAKEN ──
from src.functional import generate_with_patch

MAX_NEW = 320
ASSIGN_RE = re.compile(r"^\s{4}([a-z])\s*=(?!=)", re.M)   # 4-space indent only: skips loop bodies

def emitted_order(code):
    seen, out = set(), []
    for m in ASSIGN_RE.finditer(code):
        v = m.group(1)
        if v not in seen:
            seen.add(v); out.append(v)
    return out

for p in PROMPTS:
    gen = generate_with_patch(mt=mt, inputs=p["prompt"], max_new_tokens=MAX_NEW,
                              do_sample=False, remove_prefix=True)[0].split("</code>")[0]
    p["gen"] = gen
    order = emitted_order(gen)
    listing = [chr(97 + i) for i in range(len(p["conds"]))]
    dep_s = ", ".join(f"{c}<-{pr}" for c, pr in p["dep"]) or "—"
    # Did the consumer actually REFERENCE the producer's variable? An implicit edge is only an
    # opportunity; a taken opportunity is a dependency, an ignored one is a re-computation.
    taken = []
    for cons, prod in p["dep"]:
        m = re.search(rf"^\s{{4}}{cons}\s*=(?!=)(.*(?:\n(?!\s{{4}}[a-z]\s*=).*)*)", gen, re.M)
        taken.append(f"{cons}<-{prod}: "
                     + ("REUSED" if m and re.search(rf"\b{prod}\b", m.group(1)) else "inlined"))
    hdr = (f"[bold]{p['id']}[/bold]   {p['note']}\n"
           f"listing {' '.join(listing)}   ·   emitted [bold cyan]{' '.join(order) or '—'}[/bold cyan]"
           f"   ·   edges {dep_s}" + (f"   ·   {' | '.join(taken)}" if taken else ""))
    rich_print(Panel(Group(Text.from_markup(hdr), Text(""),
                           Text("\n".join(f"{i+1}. {c}" for i, c in enumerate(p["conds"])),
                                style="grey70"),
                           Text(""), Text(gen.strip(), style="sea_green3")),
                     border_style="cyan", padding=(1, 1)))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Q0_control   TRANSFER CHECK: same multi-line producer, but nothing depends on it                                │
│ listing a b c   ·   emitted a b c   ·   edges —                                                                 │
│                                                                                                                 │
│ 1. the number of times each word appears in words                                                               │
│ 2. whether every word is entirely lowercase                                                                     │
│ 3. whether words has at least three entries                                                                     │
│                                                                                                                 │
│ a = {word: words.count(word) for word in set(words)}                                                            │
│     b = all(word.islower() for word in words)                                                                   │
│     c = len(words) >= 3                                                                                         │
│     print(a, b, c)                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Q1_counts   b needs the counts a builds — 'any word' is unphraseable without them                               │
│ listing a b c   ·   emitted a b c   ·   edges b<-a   ·   b<-a: REUSED                                           │
│                                                                                                                 │
│ 1. the number of times each word appears in words                                                               │
│ 2. whether any word appears more than twice                                                                     │
│ 3. whether every word is entirely lowercase                                                                     │
│                                                                                                                 │
│ a = {word: words.count(word) for word in set(words)}                                                            │
│     b = any(count > 2 for count in a.values())                                                                  │
│     c = all(word.islower() for word in words)                                                                   │
│     print(a, b, c)                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Q2_two_consumers   two consumers share the counts; rebuilding twice would be absurd                             │
│ listing a b c   ·   emitted a b c   ·   edges b<-a, c<-a   ·   b<-a: REUSED | c<-a: REUSED                      │
│                                                                                                                 │
│ 1. the number of times each word appears in words                                                               │
│ 2. whether any word appears more than twice                                                                     │
│ 3. whether the most frequently occurring word is longer than four characters                                    │
│                                                                                                                 │
│ a = {word: words.count(word) for word in set(words)}                                                            │
│     b = any(count > 2 for count in a.values())                                                                  │
│     c = len(max(a, key=a.get)) > 4                                                                              │
│     print(a, b, c)                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Q3_process   STRONGEST: b depends on a's PROCESS, not its value — inlining means duplicating the loop           │
│ listing a b c   ·   emitted a b c m   ·   edges b<-a   ·   b<-a: inlined                                        │
│                                                                                                                 │
│ 1. the running maximum word length after each word in words                                                     │
│ 2. how many times that running maximum increased                                                                │
│ 3. whether every word is entirely lowercase                                                                     │
│                                                                                                                 │
│ a = []                                                                                                          │
│     b = 0                                                                                                       │
│     c = True                                                                                                    │
│     m = 0                                                                                                       │
│     for w in words:                                                                                             │
│         ml = len(w)                                                                                             │
│         if ml > m:                                                                                              │
│             m = ml                                                                                              │
│             b += 1                                                                                              │
│         a.append(m)                                                                                             │
│         if not w.islower():                                                                                     │
│             c = False                                                                                           │
│     print(a, b, c)                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Q4_positional   positional consumer — b indexes into the ordering a produces                                    │
│ listing a b c   ·   emitted a b c   ·   edges b<-a   ·   b<-a: REUSED                                           │
│                                                                                                                 │
│ 1. the words sorted by length, shortest first                                                                   │
│ 2. whether the third of those is longer than four characters                                                    │
│ 3. whether every word is entirely lowercase                                                                     │
│                                                                                                                 │
│ a = sorted(words, key=len)                                                                                      │
│     b = len(a[2]) > 4 if len(a) > 2 else False                                                                  │
│     c = all(word.islower() for word in words)                                                                   │
│     print(a, b, c)                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 3. Causal — move-on at every rung of the producer's implementation

One **ladder** per prompt: the same producer, prefilled to successively later points in its own
implementation. A rung is *how far along* the sub-step is when the pivot lands.

The clean run tells you which rungs are real tests:

* while the value is **unfinished**, the clean run should **stay** — extend or re-assign the same
  letter. Move-on there means abandoning a half-built value, which is the thing worth measuring.
* at the **last** rung the value is done, so the clean run should move on *by itself*. That rung is
  the **negative control**: clean and patched agreeing there is expected, not a failure.

`stays` records the expected clean continuation for each rung, so a broken prefill is visible as a
mismatch rather than being read as a result. `Q0` is also the OOD transfer check — the move-on vector
is cached on the `int` / single-line corpus, and these prompts are `list[str]` and chunky.

In [5]:
# ── §3: intervene at an incomplete sub-step ────────────────────────────────
from src.goals.head_loc_attn import HeadMask, get_o_proj_inputs_batch
from src.goals.completion_pairs import build_completion_pairs
from src.goals.goalscope import headpatched_generate
from src.goals.oid_gaze import cache_root
from src.tokens import prepare_input
from src.goals.multiline_prompts import replace_in_code_newline_with_tabnewline

VERSION, TAG, SEL_THR, ALPHA = "conditions", "lam10.0", 0.8, 8.0
_root = os.path.join(cache_root(MODEL_NAME, VERSION), "completion_mask")
_c = [t for t in os.listdir(_root) if os.path.isdir(os.path.join(_root, t)) and TAG in t]
DCK = torch.load(os.path.join(_root, max(_c, key=lambda t: os.path.getmtime(os.path.join(_root, t))),
                              "mask.pt"), map_location="cpu", weights_only=False)
dec_layers = [int(x) for x in DCK["layers"]]
decision_heads = [(dec_layers[li], int(h)) for li, h in np.argwhere(DCK["mask_values"].numpy() > SEL_THR)]
PATCH_POS = int(DCK["patch_tokens"][0]) if len(DCK.get("patch_tokens", [-2])) == 1 else -2

def mask_bin(heads):
    m = HeadMask(mt.n_layer, mt.model.config.num_attention_heads)
    with torch.no_grad():
        m.values.zero_()
        for (l, h) in heads: m.values[l, h] = 1.0
    return m
DEC_MASK, NULL_MASK = mask_bin(decision_heads), mask_bin([])

# The move-on vector is cached on the INT / single-line corpus the mask was trained on — these
# prompts are list[str] and chunky, so Q0 below is the check that it transfers at all.
_pairs = build_completion_pairs(mt, n_pairs=256, target_slots=(1, 2, 3), direction="sl->ml",
                                same=True, patch_tokens=(PATCH_POS,), seed=1234, log=logger.debug)
_acc = {l: [] for l in dec_layers}
for i in range(0, len(_pairs), 16):
    cache = get_o_proj_inputs_batch(mt, [p.source.sl_prompt for p in _pairs[i:i+16]],
                                    dec_layers, token_positions=[PATCH_POS])
    for l in dec_layers: _acc[l].append(cache[l][:, 0, :].float().cpu())
MEAN_SRC = torch.stack([torch.cat(_acc[l], 0).mean(0) for l in dec_layers])
print(f"decision heads {len(decision_heads)} · PATCH_POS={PATCH_POS} · vector {tuple(MEAN_SRC.shape)}")

decision heads 181 · PATCH_POS=-2 · vector (64, 5120)


In [6]:
# ── §3 cases: one ladder per prompt, pivot at each rung ─────────────────────
# (rung label, cumulative body lines, expected clean continuation, value-is-complete)
COUNTS = [
    ("declared",   ["    a = {}"],
     "for w in words:", False),
    ("loop open",  ["    a = {}", "    for w in words:"],
     "a[w] = a.get(w, 0) + 1", False),
    ("complete",   ["    a = {}", "    for w in words:", "        a[w] = a.get(w, 0) + 1"],
     "(value done — clean should move on by itself)", True),
]
RUNMAX = [
    ("declared",   ["    a = []"],
     "m = 0", False),
    ("accum init", ["    a = []", "    m = 0"],
     "for w in words:", False),
    ("loop open",  ["    a = []", "    m = 0", "    for w in words:"],
     "m = max(m, len(w))", False),
    ("mid body",   ["    a = []", "    m = 0", "    for w in words:",
                    "        m = max(m, len(w))"],
     "a.append(m)", False),
    ("complete",   ["    a = []", "    m = 0", "    for w in words:",
                    "        m = max(m, len(w))", "        a.append(m)"],
     "(value done — clean should move on by itself)", True),
]
SORTING = [
    ("copied",     ["    a = list(words)"],
     "a.sort(key=len)", False),
    ("complete",   ["    a = list(words)", "    a.sort(key=len)"],
     "(value done — clean should move on by itself)", True),
]
LADDERS = {                      # prompt -> which producer it builds
    "Q0_control":       COUNTS,  # control: nothing downstream needs `a`
    "Q1_counts":        COUNTS,
    "Q2_two_consumers": COUNTS,
    "Q3_process":       RUNMAX,  # consumer needs a's PROCESS, not just its value
    "Q4_positional":    SORTING,
}
CASES = [dict(pid=pid, depth=d, body=b, stays=s, complete=c, pivot="a")
         for pid, ladder in LADDERS.items() for (d, b, s, c) in ladder]
print(f"{len(CASES)} cases over {len(LADDERS)} prompts "
      f"({sum(not c['complete'] for c in CASES)} real tests, "
      f"{sum(c['complete'] for c in CASES)} negative controls)")
for pid, ladder in LADDERS.items():
    print(f"  {pid:18s} {len(ladder)} rungs: " + " -> ".join(d for d, *_ in ladder))

16 cases over 5 prompts (11 real tests, 5 negative controls)
  Q0_control         3 rungs: declared -> loop open -> complete
  Q1_counts          3 rungs: declared -> loop open -> complete
  Q2_two_consumers   3 rungs: declared -> loop open -> complete
  Q3_process         5 rungs: declared -> accum init -> loop open -> mid body -> complete
  Q4_positional      2 rungs: copied -> complete


In [7]:
# ── §3 run: clean vs move-on-patched at every rung ──────────────────────────
def show_case(case, max_new=48):
    p = next(x for x in PROMPTS if x["id"] == case["pid"])
    code = STUB + "\n".join(case["body"]) + "\n"
    user = p["prompt"].split("<|im_start|>user")[-1].split("<|im_end|>")[0].strip()
    full = apply_chat_template(mt=mt, user_query=user, incomplete_code=code)
    prompt = replace_in_code_newline_with_tabnewline(full) + "   "
    inp = prepare_input(prompts=prompt, tokenizer=mt)
    ids = inp.input_ids[0].tolist()
    pivot_tok = mt.tokenizer.decode([ids[len(ids) + PATCH_POS]])
    clean = headpatched_generate(mt, dict(inp), MEAN_SRC, NULL_MASK, dec_layers,
                                 max_new_tokens=max_new, patch_input_positions=[PATCH_POS])
    patched = headpatched_generate(mt, dict(inp), MEAN_SRC, DEC_MASK, dec_layers,
                                   max_new_tokens=max_new, n_steer_tokens=1,
                                   patch_input_positions=[PATCH_POS], alpha=ALPHA)
    dep_s = ", ".join(f"{c}<-{pr}" for c, pr in p["dep"]) or "—"
    kind = ("[grey58]NEGATIVE CONTROL — value complete, clean is expected to move on[/grey58]"
            if case["complete"] else
            f"[grey58]clean should STAY, i.e. write something like: {case['stays']}[/grey58]")
    rich_print(Panel(
        Group(Text.from_markup(
                  f"[bold]{case['pid']}[/bold]  ·  rung [yellow]{case['depth']}[/yellow]"
                  f"  ·  edges {dep_s}  ·  pivot token {pivot_tok!r}\n{p['note']}\n{kind}"),
              Text(""),
              Text("\n".join(f"{i+1}. {c}" for i, c in enumerate(p["conds"])), style="grey70"),
              Text(""),
              Text("prefill (pivot = end of the last line):", style="bold"),
              Text(code.rstrip(), style="white")),
        border_style=("grey42" if not case["complete"] else "grey30"), padding=(1, 1)))
    t = Table(box=SIMPLE_HEAVY, expand=True, pad_edge=False)
    t.add_column("clean  (no patch)", style="sea_green3", ratio=1, overflow="fold")
    t.add_column("patched  (move-on injected)", style="salmon1", ratio=1, overflow="fold")
    t.add_row(escape(clean.strip()), escape(patched.strip()))
    rich_print(t); rich_print("")
    return dict(pid=case["pid"], rung=case["depth"], complete=case["complete"],
                clean=clean.strip(), patched=patched.strip())

RESULTS = [show_case(c) for c in CASES]
print(f"{len(RESULTS)} generations — see RESULTS for the raw text")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Q0_control  ·  rung declared  ·  edges —  ·  pivot token '    \n'                                               │
│ TRANSFER CHECK: same multi-line producer, but nothing depends on it                                             │
│ clean should STAY, i.e. write something like: for w in words:                                                   │
│                                                                                                                 │
│ 1. the number of times each word appears in words                                                               │
│ 2. whether every word is entirely lowercase                                                                     │
│ 3. whether words has at least three entries                                                                     │
│                                                                                                                 │
│ prefill (pivot = end of the last line):                                                                         │
│ def check_conditions(words: list[str]):                                                                         │
│     a = {}                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean  (no patch)                                         patched  (move-on injected)                             
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 for word in words:                                        for word in words: a[word] = a.get(word, 0) + 1         
         a[word] = a.get(word, 0) + 1                          b = all(word.islower() for word in words)           
     b = all(word.islower() for word in words)                 c = len(words) >= 3                                 
     c = len(words) >= 3                                       print(a,                                            
     print                                                                                                        

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Q0_control  ·  rung loop open  ·  edges —  ·  pivot token '    \n'                                              │
│ TRANSFER CHECK: same multi-line producer, but nothing depends on it                                             │
│ clean should STAY, i.e. write something like: a = a.get(w, 0) + 1                                               │
│                                                                                                                 │
│ 1. the number of times each word appears in words                                                               │
│ 2. whether every word is entirely lowercase                                                                     │
│ 3. whether words has at least three entries                                                                     │
│                                                                                                                 │
│ prefill (pivot = end of the last line):                                                                         │
│ def check_conditions(words: list[str]):                                                                         │
│     a = {}                                                                                                      │
│     for w in words:                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean  (no patch)                                         patched  (move-on injected)                             
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 a[w] = a.get(w, 0) + 1                                    b = all(w.islower() for w in words)                     
     b = all(w.islower() for w in words)                       c = len(words) >= 3                                 
     c = len(words) >= 3                                       print(a, b, c)                                      
     print(a, b, c)                                        </code>                                                

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Q0_control  ·  rung complete  ·  edges —  ·  pivot token '    \n'                                               │
│ TRANSFER CHECK: same multi-line producer, but nothing depends on it                                             │
│ NEGATIVE CONTROL — value complete, clean is expected to move on                                                 │
│                                                                                                                 │
│ 1. the number of times each word appears in words                                                               │
│ 2. whether every word is entirely lowercase                                                                     │
│ 3. whether words has at least three entries                                                                     │
│                                                                                                                 │
│ prefill (pivot = end of the last line):                                                                         │
│ def check_conditions(words: list[str]):                                                                         │
│     a = {}                                                                                                      │
│     for w in words:                                                                                             │
│         a[w] = a.get(w, 0) + 1                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean  (no patch)                                         patched  (move-on injected)                             
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 b = all(w.islower() for w in words)                       b = all(w.islower() for w in words)                     
     c = len(words) >= 3                                       c = len(words) >= 3                                 
     print(a, b, c)                                            print(a, b, c)                                      
 </code>                                                   </code>                                                

... (23 outputs left out)

clean  (no patch)                                         patched  (move-on injected)                             
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 a.sort(key=len)                                           a.sort(key=len)                                         
     b = len(a[2]) > 4                                         b = len(a[2]) > 4 if len(a) > 2 else False          
     c = all(w.islower() for w in words)                       c = all(word.islower() for word in words)           
     print(a, b, c)                                            print(a, b, c)                                      
 </code>                                                   </                                                     

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Q4_positional  ·  rung complete  ·  edges b<-a  ·  pivot token '    \n'                                         │
│ positional consumer — b indexes into the ordering a produces                                                    │
│ NEGATIVE CONTROL — value complete, clean is expected to move on                                                 │
│                                                                                                                 │
│ 1. the words sorted by length, shortest first                                                                   │
│ 2. whether the third of those is longer than four characters                                                    │
│ 3. whether every word is entirely lowercase                                                                     │
│                                                                                                                 │
│ prefill (pivot = end of the last line):                                                                         │
│ def check_conditions(words: list[str]):                                                                         │
│     a = list(words)                                                                                             │
│     a.sort(key=len)                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean  (no patch)                                         patched  (move-on injected)                             
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 b = len(a[2]) > 4                                         b = len(a[2]) > 4 if len(a) > 2 else False              
     c = all(w.islower() for w in words)                       c = all(w.islower() for w in words)                 
     print(a, b, c)                                            print(a, b, c)                                      
 </code>                                                   </code>                                                

16 generations — see RESULTS for the raw text
